In [1]:
####################################
# Packages
####################################

!pip install --quiet ecos
import cvxpy as cp
import numpy as np

In [2]:
####################################
# Auxiliary functions
####################################

# Function that builds a set of outbound and inbound sailing legs for each route
def build_legs(routes):
  legs = {}

  N_route_ports = { key:np.size(routes[key]) for key in routes.keys() }
  for s in routes.keys():
    legs[s] = set()

    for k in range(N_route_ports[s]-1):
      i = routes[s][k]; j = routes[s][k+1]
      legs[s].add((i, j)); legs[s].add((j, i))

  return legs

# Function that builds a set of active origin-destination cargo flow arcs for each leg in each route
def build_leg_flow_arcs(routes):
  leg_flow_arcs = {}

  N_route_ports = { key:np.size(routes[key]) for key in routes.keys() }
  for s in np.arange(len(routes)):  # change to routes.keys()
    route_arcs = {}
    for k in range(N_route_ports[s]-1):
      arcs = []
      i = routes[s][k]; j = routes[s][k+1]
      # Loop from the initial departure port to the departure port of the current leg
      for k_o in range(0, k+1):
        # Loop from the current departure port to the final arrival port
        for k_d in range(k+1, N_route_ports[s]):
          i_p = routes[s][k_o]; j_p = routes[s][k_d]
          arcs.append((i_p, j_p))
      route_arcs[(i,j)] = arcs
    leg_flow_arcs[s] = route_arcs

  return leg_flow_arcs

# Function that builds a set of origin-destination cargo flow arcs for the transportation network
def build_flow_arcs(leg_demand_arcs):
  flow_arcs = {}
  for s in leg_demand_arcs.keys():
    flow_arcs[s] = set()
    for leg in leg_demand_arcs[s].keys():
      for (i, j) in leg_demand_arcs[s][leg]:
        flow_arcs[s].add((i, j)); flow_arcs[s].add((j, i))

  return flow_arcs

# Function that builds sets of origin-destination cargo flow arcs that load and offload
# at the destination port of each sailing leg in each route
def build_destination_flow_arcs(routes):
  N_route_ports = { key:np.size(routes[key]) for key in routes.keys() }

  offload_arcs = {}
  load_arcs = {}

  for s in routes.keys():
    route_offload_arcs = {}
    route_load_arcs = {}

    for k in range(N_route_ports[s]-1):
      i_p = routes[s][k]
      j_p = routes[s][k+1]
      ib_offload = []; ib_load = [];
      ob_offload = []; ob_load = [];

      # OUTBOUND LEG
      # Enumerate o-d pairs that offload at the destination port
      for k_o in range(0, k+1):
        ob_offload.append((routes[s][k_o], routes[s][k+1]))
      # Enumerate o-d pairs that load at the destination port
      for k_d in range(k+2, N_route_ports[s]):
        ob_load.append((routes[s][k+1], routes[s][k_d]))
      # Check separately the case where the destination port is the last port
      if k == N_route_ports[s]-2:
        # Iterate all the o-d pairs originating from the last port.
        for k_d in range(0, N_route_ports[s]-1):
          ob_load.append((routes[s][k+1], routes[s][k_d]))

      # INBOUND LEG
      # Enumerate o-d pairs that offload at the destination port
      for k_o in range(k+1, N_route_ports[s]):
        ib_offload.append((routes[s][k_o], routes[s][k]))

      # Enumerate o-d pairs that load at the destination port
      for k_d in range(0, k):
        ib_load.append((routes[s][k], routes[s][k_d]))
      # Check separately the case where the destination port is the last port
      if k == 0:
        # Iterate all the o-d pairs originating from the last port
        for k_d in range(1, N_route_ports[s]):
          ib_load.append((routes[s][0], routes[s][k_d]))

      route_offload_arcs[(i_p, j_p)] = ob_offload
      route_offload_arcs[(j_p, i_p)] = ib_offload
      route_load_arcs[(i_p, j_p)] = ob_load
      route_load_arcs[(j_p, i_p)] = ib_load

    offload_arcs[s] = route_offload_arcs
    load_arcs[s] = route_load_arcs

  return offload_arcs, load_arcs

In [3]:
####################################
# CVXPY implementation
####################################

def build_and_solve(ship_params, network_params):

  #
  # Data
  #

  beta, eta_el, gamma_batt, k_A, N_sup, P_aux_unit, A_sup_unit, beta_KB, eps_KG, sigma_y_max, k_safe, \
  rho_vol, rho_mass, rho_st, p_sup, h_sup, h_roro, h_batt_min, l_CB, E_a, R_g, T_cell, E_tilde, chi, \
  V_cell, rho_max, C_st, C_batt, t_ship, V_int_hat, k_vol, N_crew_const, N_crew_var, C_crew \
  = map(ship_params.get, ['beta', 'eta_el', 'gamma_batt', 'k_A', 'N_sup', 'P_aux_unit', 'A_sup_unit',
                            'beta_KB', 'eps_KG', 'sigma_y_max', 'k_safe', 'rho_vol', 'rho_mass', 'rho_st',
                            'p_sup', 'h_sup', 'h_roro', 'h_batt_min', 'l_CB', 'E_a', 'R_g', 'T_cell',
                            'E_tilde', 'chi', 'V_cell', 'rho_max', 'C_st', 'C_batt', 't_ship', 'V_int_hat',
                            'k_vol', 'N_crew_const', 'N_crew_var', 'C_crew'
                            ])

  U_min, U_bias, t_load, C_el, C_ch, C_port, P_cha_max, L_od, f_dem, inactive_flow_arcs, \
  routes, t_route, fixed_fleet, uniform_fleet, present_fleet, t_port_min, labels \
  = map(network_params.get, ['U_min', 'U_bias', 't_load', 'C_el', 'C_ch', 'C_port', 'P_cha_max', 'L_od',
                             'f_dem', 'inactive_flow_arcs', 'routes', 't_route', 'fixed_fleet',
                             'uniform_fleet', 'present_fleet', 't_port_min', 'labels'])

  #
  # Indexing sets
  #

  # Ports
  N_route_ports = { key:np.size(routes[key]) for key in routes.keys() }
  N_ports = max([max(v) for v in routes.values()])+1
  ports = np.arange(N_ports)

  # Ships
  N_services = len(routes)
  services = np.arange(N_services)

  # Cargoes
  cargoes = np.arange(2)

  # Sailing legs
  leg_demand_arcs = build_leg_flow_arcs(routes)
  flow_arcs = build_flow_arcs(leg_demand_arcs)
  legs = build_legs(routes)
  offload_flow_arcs, load_flow_arcs = build_destination_flow_arcs(routes)

  # Demand origin-destination pairs and sailing legs
  active_od = set()

  for s in services:
    active_od = active_od.union(flow_arcs[s])

  #
  # Decision variables
  #

  q = {}        # cargo quantity shipped, 1000 pax or 1000 lane-m
  v = {}        # speed, km/h
  t_port = {}   # port turnaround time, h
  r_Cf = {}     # auxiliary variable for frictional resistance
  r_Cr_std = {} # auxiliary variable for residual resistance

  for s in services:
    for (i, j) in flow_arcs[s]:
      for c in cargoes:
        q[(i, j, s, c)] = cp.Variable(name=f"q_{i}_{j}_{s}_{c}", pos=True)

    for (i, j) in legs[s]:
      v[(i, j, s)] = cp.Variable(name=f"v_{i}_{j}_{s}", pos=True)
      t_port[(i, j, s)] = cp.Variable(name=f"t_port_{i}_{j}_{s}", pos=True)
      r_Cf[(i, j, s)] = cp.Variable(name=f"r_Cf_{i}_{j}_{s}", pos=True)
      r_Cr_std[(i, j, s)] = cp.Variable(name=f"r_Cr_std_{i}_{j}_{s}", pos=True)

  N_trip = cp.Variable(N_services, name="N_trip", pos=True)           # service frequency
  N_ship = cp.Variable(N_services, name="N_ship", pos=True)           # num of ships in a service

  L_sup = cp.Variable(N_services, name="L_sup", pos=True)             # superstructure length, m
  L = cp.Variable(N_services, name="L", pos=True)                     # hull length, m
  B = cp.Variable(N_services, name="B", pos=True)                     # hull waterline breadth, m
  T = cp.Variable(N_services, name="T", pos=True)                     # draft, m
  D = cp.Variable(N_services, name="D", pos=True)                     # hull depth (keel to topmost continuous deck), m
  p_td = cp.Variable(N_services, name="p_td", pos=True)               # deck plate thickness, m
  h_batt = cp.Variable(N_services, name="h_b", pos=True)              # battery deck height, m
  w_batt = cp.Variable((3, N_services), name="w_batt", pos=True)      # battery space width, m
  l_batt = cp.Variable((3, N_services), name="l_batt", pos=True)      # battery space length, m
  hp_batt = cp.Variable(N_services, name="hp_b", pos=True)            # battery space deck floor elevation from keel, m
  lp_batt = cp.Variable((3, N_services), name="lp_batt", pos=True)    # battery space front wall distance from origin (bow), m
  hp_roro = cp.Variable(N_services, name="hp_roro", pos=True)         # first roro deck elevation from keel, m
  P_cha = cp.Variable(N_ports, name="P_cha", pos=True)                # charger power, MW
  E_batt = cp.Variable(N_services, name="E_batt", pos=True)           # battery capacity, MJ
  t_cell = cp.Variable(N_services, name="t_cell", pos=True)           # cell cycle life, years
  N_batt = cp.Variable(N_services, name="N_batt", pos=True)           # number of batteries installed during lifetime
  r_g = cp.Variable((4, N_services), name="r_g", pos=True)            # auxiliary variable for wetted area numerical integration
  r_g2 = cp.Variable((4, N_services), name="r_g2", pos=True)          # auxiliary variable for hull girder steel area area numerical integration

  #
  # Derived quantities
  #

  # Resistance
  k_BT = [(B[s]/T[s])**0.2748 for s in services]                      # beam–draft ratio factor
  k_LB = [(L[s]/B[s])**-0.5747 for s in services]                     # length–beam ratio factor
  k_Pr = 0.7**-0.1418                                                 # propeller factor
  k_rud = 2**-0.1258                                                  # rudder factor
  k_L  = [1.8319*(L[s]**-0.1237) for s in services]                   # length factor

  # Hull
  C_B = (0.5 - np.log(beta+1)/(2*beta) + beta/(3+3*beta))             # block coefficient
  nabla = [C_B*L[s]*B[s]*T[s] for s in services]                      # displacement, m^3
  h_KG = [0.7*D[s] for s in services]                                 # vertical coordinate of hull mass center, m
  V_int = [L[s]*B[s]*D[s]*k_vol + B[s]*L_sup[s]*(N_sup*h_sup +\
           h_roro) for s in services]                                 # internal volume, t^3
  V_GT = [V_int[s]*(0.2 + 0.02*np.log10(V_int_hat)*cp.power(V_int[s]/V_int_hat,\
          1/(np.log10(V_int_hat)*np.log(10)) )) for s in services]    # gross tonnage
  B_deck = [B[s]*cp.power(D[s]/T[s], 1/beta) for s in services]       # hull width at girder top deck, m
  A_wet = [2*L[s]*k_A*cp.sum([[1, 3, 3, 1][i]*cp.power(r_g[i, s], 0.5) \
                              for i in range(4) ]) for s in services] # hull wetted area, m^2
  A_girder = [2*L[s]*k_A*cp.sum([[1, 3, 3, 1][i]*cp.power(r_g2[i, s], 0.5) \
              for i in range(4) ]) for s in services]                 # external hull steel area, m^2
  q_cap = {0: [L[s]*B[s]/(1E4) for s in services],                    # cargo capacity
       1: [N_sup*B[s]*L_sup[s]/(A_sup_unit*(1E3)) for s in services]}
  M_max = [9.81*C_B*cp.power(L[s], 2)*B[s]*T[s]/(2*(np.pi**2)*1000)\
           for s in services]                                         # max bending moment, MNm
  p_sp = [B[s]*p_td[s]/(2*D[s]) for s in services]                    # side plating thickness, m

  # Energy
  P_aux = [P_aux_unit*N_sup*B[s]*L_sup[s] for s in services]          # hotel power, kW

  # Masses
  W_side = [rho_st*p_sp[s]*A_girder[s] for s in services]             # external plating mass, t
  W_bottom = [rho_st*p_td[s]*L[s]*B[s]/2 for s in services]           # bottom plating mass, t
  W_deck = [rho_st*p_td[s]*2*5*B[s]*L[s]/6 for s in services]         # 2x roro deck mass, t
  W_tt = [rho_st*p_td[s]*2*B[s]*L[s]*((hp_batt[s]/T[s])**(1/beta))/3 \
          for s in services]                                          # tank top (battery space) deck plate mass, t
  W_tbh0 = [2*rho_st*p_sup*(beta/(beta+1))*hp_roro[s]*B[s]*((hp_roro[s]/T[s])**(1/beta))*((2*lp_batt[0, s]/L[s])**(0.5)) \
            for s in services]                                        # battery space transverse bulkheads (battery space 0) mass, t
  W_tbh1 = [2*rho_st*p_sup*(beta/(beta+1))*hp_roro[s]*B[s]*((hp_roro[s]/T[s])**(1/beta))*((2*lp_batt[1, s]/L[s])**(0.5)) \
            for s in services]                                        # battery space transverse bulkheads (battery space 1) mass, t
  W_tbh2 = [2*rho_st*p_sup*(beta/(beta+1))*hp_roro[s]*B[s]*((hp_roro[s]/T[s])**(1/beta))*((2*lp_batt[2, s]/L[s])**(0.5)) \
            for s in services]                                        # battery space transverse bulkheads (battery space 2) mass, t
  W_bwall = [rho_st*p_sup*h_batt[s]*(l_batt[0, s] + 2*l_batt[1, s] + 2*l_batt[2, s]) \
             for s in services]                                       # battery space wall mass, t
  W_bh = [rho_st*2*p_sp[s]*0.7*D[s]*L[s] for s in services]           # longitudinal bulkhead mass, t
  W_ramp = [rho_st*p_sup*B[s]*h_roro for s in services]               # stern roro ramp

  W_sup = [rho_st*p_sup*( N_sup*(2*L_sup[s]*h_sup + L_sup[s]*B[s] + 2*B[s]*h_sup) + 2*L_sup[s]*h_roro + \
           L_sup[s]*B[s] + B[s]*h_roro ) for s in services]           # superstructure plate mass, t
  W_plate = [W_side[s] + W_bottom[s] + W_bwall[s] + W_tbh0[s] + W_tbh1[s] + W_tbh2[s] + W_tt[s] + W_deck[s] + \
             W_ramp[s] + W_bh[s] + W_sup[s] for s in services]        # total plate mass
  W_fit = [0.2*(2*5*B[s]*L[s]/6 + N_sup*(B[s]*L_sup[s])) \
           for s in services]                                         # outfitting mass, t
  W_batt = [E_batt[s]*rho_mass/rho_vol for s in services]             # battery mass, t
  W_roro = [36*1000*q_cap[0][s]/23 for s in services]                 # cargo mass, t (23 m long vehicle that weighs 36 tons)
  W_sum = [1.1*W_plate[s] + W_fit[s] + W_batt[s] + W_roro[s] for s in services]

  # Leg-specific quantities
  R_fric = {}                                                         # frictional resistance, kN
  Fr = {}                                                             # Froude number
  R_res = {}                                                          # residual resistance, kN
  E_leg = {}                                                          # leg energy consumption, MJ (kN*km)
  for s in services:
    for (i, j) in legs[s]:
      R_fric[(i, j, s)] = 0.5*75*cp.power(r_Cf[(i, j, s)], -2)*A_wet[s]*((v[(i, j, s)]/3.6)**2)/1000
      Fr[(i, j, s)] = (v[(i, j, s)]/3.6)*((L[s]*9.81)**-0.5)
      R_res[(i, j, s)] = 0.5*r_Cr_std[(i, j, s)]*k_L[s]*k_BT[s]*k_LB[s]*k_Pr*k_rud*((v[(i, j, s)]/3.6)**2)*B[s]*T[s]/10
      E_leg[(i, j, s)] = (R_fric[(i, j, s)] + R_res[(i, j, s)])*L_od[i, j]/eta_el + (L_od[i,j]/v[(i,j,s)])*3.6*P_aux[s]

  L_od_norm = L_od/np.max(L_od)                                       # normalized sailing distance

  # Battery
  N_cell = [E_batt[s]/(3600*V_cell*E_tilde) for s in services]        # number of li-ion cells, 1E6
  r_dis = [N_trip[s]*2*cp.sum([E_leg[(i, j, s)]/(3600*V_cell*N_cell[s]) for (i, j) in legs[s]]) \
           for s in services]                                         # NOTE: voyage duration cancels out, because it is also present in denominator of the cell current expression
  k_batt = [N_batt[s]**-0.7 for s in services]                        # correction factor for battery installation cost
  xi = (chi[1] + chi[2]/2)*np.exp(-E_a/(R_g*T_cell))                  # degredation model constant term

  #
  # Network constraints
  #

  constr = [ ]

  # Restrict N_trip and N_ship to a set of valid integer-valued frequencies
  for s in services:
    integer_N_trip = cp.FiniteSet(N_trip[s], np.array([1, 2, 4, 6, 8]))
    constr += [integer_N_trip]
    integer_N_ship = cp.FiniteSet(N_ship[s], np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]))
    constr += [integer_N_ship]

  # Lower bounds, NOTE: NOT NEEDED IF INTEGER CONSTRAINTS ARE ACTIVE
  #for s in services:
    #constr += [ N_trip[s] >= 1, N_ship[s] >= 1 ]

  # Fix fleet composition
  if present_fleet:
    for s in services:
      constr += [N_trip[s] == fixed_fleet[s]['N_trip'],
                 N_ship[s] == fixed_fleet[s]['N_ship']]

  # Ship capacity
  for s in services:
    for leg in leg_demand_arcs[s].keys(): # iterate legs (outbound) and impose a constr for each
      for c in cargoes:
        constr += [ cp.sum([q[(i, j, s, c)] for (i, j) in leg_demand_arcs[s][leg]]) <= q_cap[c][s]*N_trip[s]*N_ship[s],
                    cp.sum([q[(j, i, s, c)] for (i, j) in leg_demand_arcs[s][leg]]) <= q_cap[c][s]*N_trip[s]*N_ship[s] ]

  # Port turnaround time
  for s in services:
    for (i, j) in legs[s]:
      port_jobs = []
      for c in cargoes:
        offload_sum = cp.sum([q[(i_p, j_p, s, c)] for (i_p, j_p) in offload_flow_arcs[s][(i, j)]])/t_load[c]
        load_sum = cp.sum([q[(i_p, j_p, s, c)] for (i_p, j_p) in load_flow_arcs[s][(i, j)]])/t_load[c]
        # NOTE: this formulation implies that the loading and offloading are SEQUENTIAL events
        port_jobs.append(offload_sum + load_sum);
      port_jobs.append(E_leg[(i, j, s)]/(P_cha[j]*3600))  # battery charging, NOTE THE INDEXING
      constr += [  t_port[(i, j, s)] >= cp.max(cp.vstack(port_jobs))/(N_trip[s]*N_ship[s]) ]
      if present_fleet:
        constr += [ t_port[(i, j, s)] >= t_port_min[s][j] ]

  # Availability
  for s in services:
    # Operating hours do not exceed ship availability
    constr += [ N_trip[s]*cp.sum([(L_od[i,j]/v[(i,j,s)] + t_port[(i,j,s)]) for (i,j) in legs[s]]) <= t_route[s] ]

  # Demand
  # calculate sum of volume shipped by all ships in all the o-d pairs
  # only enforce demand constraints for active o-d pairs
  for (i, j) in active_od:
    for c in cargoes:
      lhs = 0
      for s in services:
        if (i, j) in flow_arcs[s]:
          lhs += q[(i, j, s, c)]
      constr += [ lhs <= f_dem[c][i, j] ]

  # Charger installation power
  for i in ports:
    constr += [P_cha[i] <= P_cha_max]

  # Utility (NOTE: scaling of the values by 10)
  flow_prod = 1
  for s in services:
    for (i, j) in flow_arcs[s]:
      if (i, j) not in inactive_flow_arcs:
        for c in cargoes:
          flow_prod *= cp.power(10*q[(i, j, s, c)], U_bias[c]*L_od_norm[i, j])
  constr += [ cp.log(flow_prod) >= U_min ]

  #
  # Ship design constraints
  #

  # If the fleet is uniform, equalize principal dimensions
  if uniform_fleet:
    for s in services:
      constr += [L[s] == L[0], B[s] == B[0], T[s] == T[0], D[s] == D[0], L_sup[s] == L_sup[0], E_batt[s] == E_batt[0]]

  # If the fleet is fixed, assign principal length dimensions
  if present_fleet:
    for s in services:
      constr += [L[s] == fixed_fleet[s]['L'], L_sup[s] == fixed_fleet[s]['L_sup']]

  # Variable bounds
  for s in services:
    constr += [L_sup[s] >= 0.5*L[s], L_sup[s] <= 0.9*L[s]]

  for s in services:
    # Principal dimension bounds
    constr += [
        L[s] <= 240, L[s] >= 80,
        L[s]/(nabla[s]**(1/3)) >= 4.41,  L[s]/(nabla[s]**(1/3)) <= 1.2*7.27,
        L[s]/B[s] >= 3.96, L[s]/B[s] <= 7.13,
        B[s]/T[s] >= 2.31, B[s]/T[s]  <= 6.11
    ]

    leg_cons = []
    for (i, j)  in legs[s]:
      # Frictional resistance
      # NOTE: cvxpy log is base e, which needs to be conerted to base 10
      constr += [r_Cf[(i, j, s)] + 2 <= cp.log( (v[(i, j, s)]/3.6)*L[s]/(1.004E-6) )/np.log(10) ]
      leg_cons.append(E_leg[(i, j , s)])

      # Residual resistance
      constr += [ r_Cr_std[(i, j, s)]**1.69303 >= 258.6*(Fr[(i, j, s)]**4.83749) + 0.00929736*(Fr[(i, j, s)]**-1.44917) ]

    # Battery energy
    constr += [E_batt[s] >= cp.max(cp.vstack(leg_cons))*gamma_batt ]

    # Cell life
    constr += [rho_max >= cp.power(t_cell[s]*180*r_dis[s], chi[0])*xi*cp.exp(chi[3]*r_dis[s]/(R_g*T_cell*t_route[s]*E_tilde)) ]

    # Number of batteries installed during lifetime
    constr += [N_batt[s] >= cp.maximum(t_ship/t_cell[s], 1)]

    # Arc length integrations
    for i in range(4):  # enumerate the Simpson terms
      if i == 0:
        constr += [ r_g[i, s] >= 1,  r_g2[s, i] >= 1 ]
      else:
        constr += [ r_g[i, s] >= 1 + cp.power(2/B[s], 2*beta)*cp.power(T[s]*beta, 2)*cp.power(i*B[s]/6, 2*beta - 2),
                    r_g2[i, s] >= 1 + cp.power(2/B[s], 2*beta)*cp.power(T[s]*beta, 2)*cp.power(i*B_deck[s]/6, 2*beta - 2)]

    # Trasnverse stability
    constr += [ 0.4707*cp.power(B[s]/T[s], 1.4906) >= (h_KG[s] + eps_KG)/(beta_KB*T[s]) ]

    # Bending strength
    constr += [ k_safe*M_max[s]*90/(133*p_td[s]*B[s]*D[s]) <= sigma_y_max ]

    # Vertical force balance (buoyancy)
    constr += [nabla[s] >= W_sum[s]]

    # General arrangement
    for i in range(3):
      constr += [ w_batt[i, s]/2 <= (B[s]/2)*((2*lp_batt[i, s]/L[s])**0.5)*((hp_batt[s]/T[s])**(1/beta))]
      if i <= 1:
        constr += [ l_batt[i, s]*w_batt[i, s]*h_batt[s] == l_batt[i+1, s]*w_batt[i+1, s]*h_batt[s],   # set spaces to equal volume
                    l_batt[i+1, s] + lp_batt[i+1, s] <= lp_batt[i, s]]                                # relative space position

    constr += [ E_batt[s] <= 5*l_batt[0, s]*w_batt[0, s]*h_batt[s]*rho_vol,                           # total capacity of all battery spaces (5)
                h_batt[s] >= h_batt_min,                                                              # minimum battery space height
                hp_batt[s] >= 0.1*D[s],                                                               # minimum double bottom height
                l_CB >= (lp_batt[0, s] + l_batt[0, s]/2)/L[s],                                        # battery mass center located at the center of buouancy
                h_batt[s] + hp_batt[s] <= hp_roro[s],                                                 # first roro deck vertical position (distance from keel)
                hp_roro[s] + h_roro <= D[s],                                                          # strength deck vertical position (distance from keel)
                hp_roro[s] >= T[s] ]                                                                  # first roro deck position above waterline

  #
  # Objective
  #

  objective_func = 180*C_el*cp.sum( [N_trip[s]*N_ship[s]*E_leg[(i, j, s)] for s in services for (i,j) in legs[s]] ) \
                 + 180*cp.sum( [(V_GT[s]/1000)*N_ship[s]*N_trip[s]*C_port[j] for s in services for (i, j) in legs[s]] ) \
                 + C_crew*cp.sum([(q_cap[1][s]*N_crew_var + N_crew_const)*N_ship[s] for s in services]) \
                 + C_batt*cp.sum([N_batt[s]*k_batt[s]*E_batt[s]*N_ship[s] for s in services])/t_ship  \
                 + 10*C_st*cp.sum([N_ship[s]*W_plate[s] for s in services])/t_ship  \
                 + C_ch*cp.sum( [P_cha[i] for i in ports] )/t_ship

  #
  # Problem construction
  #

  prob = cp.Problem(cp.Minimize(objective_func), constr)
  prob.solve(gp=True, verbose=False, solver='ECOS_BB')
  print('Status:', prob.status)
  print(f"Objective [kEUR]: {prob.value:.1f} ")

  #
  # Printout
  #

  print('Charger powers [MW]: ', [f"{labels[i]}: {P_cha[i].value:.2f}" for i in ports])
  for s in services:
    print(f"Service {s}: {[labels[i] + '->' for i in routes[s]]}")
    print(f"  N_trip: {N_trip[s].value:.2f}, N_ship: {N_ship[s].value:.2f}, E_batt [MWh]: {E_batt[s].value/3600:.2f}, t_cell: {t_cell[s].value:.2f}, GT: {V_GT[s].value:.1f}. L: {L[s].value:.2f}, L_sup: {L_sup[s].value:.2f}, B: {B[s].value:.2f}, T: {T[s].value:.2f}, D: {D[s].value:.2f}, p_td: {p_td[s].value:.4f}")
    print('  Speeds: ', [f"{labels[i]}->{labels[j]}: {v[(i, j, s)].value*0.53995:.2f}" for (i, j) in legs[s]])
    """
    print(f"  Port stay")
    for (i,j) in legs[s]:
      print(f"    {i} -> {j}: {t_port[(i, j, s)].value:.2f}")
    print(f"  Volume")
    for i in ports:
      for j in range(i, N_ports):
          if (i, j) in flow_arcs[s]:
            print(f"    {i} -> {j}: {f[(i, j, s, 0)].value:.2f} ({f[(i, j, s, 1)].value:.2f}), {j} -> {i}: {f[(j, i, s, 0)].value:.2f} ({f[(j, i, s, 1)].value:.2f})")
    """


In [4]:
####################################
# Common ship data
####################################

def generate_ship_data():
  params = {
      'beta': 6,                  # hull fullness parameter
      'k_A': 0.95,                # hull wetted area correction factor
      'N_sup': 3,                 # number of decks in the superstructure
      'h_sup': 3.0,               # superstructure deck height, m
      'h_roro': 5.5,              # roro deck height, m
      'h_batt_min': 2.2,          # minimum battery space height, m
      'V_int_hat': 1.0E5,         # reference hull internal volume, m^3
      'k_vol': 0.8,               # hull volume block coefficient
      'eta_el': 0.6,              # total powertrain conversion efficiency (propeller and power electronics)
      'P_aux_unit': 0.08,         # auxiliary power, kW/m^2 accommodation floor area
      'A_sup_unit': 7.5,          # accommodation floor area per passenger, m^2 (roughly between 5 and 12)
      'beta_KB': 0.56195,         # relative vertical center of buoyancy, for beta = 6
      'eps_KG': 1.0,              # transverse stability margin
      'sigma_y_max': 235,         # maximum normal stress, MPa
      'k_safe': 2.0,              # girder stress limit safety factor
      'rho_vol': 167,             # battery space volumetric energy density, "MAERSK compact design", MJ/m^3
      'rho_mass': 0.417,          # battery space gravimetric energy density, "MAERSK compact design", t/m^3
      'rho_st': 8,                # steel density, t/m^3
      'p_sup': 0.01,              # thickness of non-girder plates (superstructure and transverse bulkheads), m
      'l_CB': 0.52394,            # relative longitudinal center of buoyancy for beta=6, m
      'E_a': 31500,               # activation energy
      'R_g': 8.314,               # gas constant
      'T_cell': 298.15,           # cell temperature, K
      'E_tilde': 2.5,             # cell capacity, Ah
      'V_cell': 3.3,              # cell voltage, V
      'chi': [0.6, 74.112,\
              28.966, 151.2],     # cell degredation fitting params
      'gamma_batt': 1.5,          # battery reserve capacity factor
      'rho_max': 0.1,             # maximum permissible cell capacity loss, %/100
      't_ship': 20,               # ship lifetime, years
      'N_crew_const': 20,         # number of fixed crew members in deck and engineering
      'N_crew_var': 146,          # number of hotel crew members per 1k pax
      'C_crew': 38,               # annual cost of an onboard crew member, kEUR
      'C_st': 0.99,               # steel cost, kEUR/t
      'C_batt': 0.0447,           # battery system cost, 2030 estimate, kEUR/MJ (50 EUR/MJ = 180 EUR/kWh)
    }

  return params

####################################
# Common network data
####################################

def generate_network_data():
  params = {
      't_load': [2.6, 10],                            # loading and unloading time, for trucks 1000 lane-m/h, for passengers 1000 pax/h
      'U_min': 80,                                    # required network utility
      'U_bias': [1, 3],                               # cargo type utility
      'C_el': 0.016667E-3,                            # electricity price, kEUR/MJ
      'C_ch': 129,                                    # charger installation cost, kEUR/MW
      'C_port': [0.1, 0.1, 0.1, 0.1, 0.1],            # port charge, kEUR/(1000 GT)
      'P_cha_max': 1000,                              # maximum charging power, MW
      'L_od': np.array([[1,  80,   1,    315,  461],  # sailing distance, km
                        [80,  1,    342,  320,  432],
                        [1,   342,  1,    160,  306],
                        [315, 320,  160,  1,    146],
                        [461, 432,  306,  146,  1]]),
      'f_dem': {0:np.array([[0.0,   47.92, 0.01,  0.01, 3.0], # demand
                            [47.92, 0.0,   0.01,  0.02, 2.21],
                            [0.01,  0.01,  0.0,   0.25, 19.85],
                            [0.01,  0.02,  0.25,  0.0,  0.32],
                            [3.0,   2.21,  19.85, 0.32, 0.0]]),
                1:np.array([[0.0,   20.50, 0.01,  0.05, 1.72],
                            [20.50, 0.0,   0.01,  0.1,  4.56],
                            [0.01,  0.01,  0.0,   1.36, 6.10],
                            [0.05,  0.1,   1.36,  0.0,  1.71],
                            [1.72,  4.56,  6.10,  1.71, 0.0]])},
      'inactive_flow_arcs': [(0, 2), (2, 0), (1, 2), (2, 1)],
      'labels': ['Tal', 'Hel', 'Tur', 'Mar', 'Sto']
    }

  return params

In [5]:
####################################
# Case B4: four routes, fixed fleet
####################################

def run_B4():
  ship_params = generate_ship_data()
  network_params = generate_network_data()

  network_params.update({ 'routes':{  0: [0, 1],        # TAL -> HEL
                                      1: [0, 3, 4],     # TAL -> MAR -> STO
                                      2: [1, 3, 4],     # HEL -> MAR -> STO
                                      3: [2, 3, 4]},    # TUR -> MAR -> STO
                          't_route': [36, 48, 48, 48],
                          'fixed_fleet': {0: {'N_trip': 6, 'N_ship': 3, 'L': 212, 'B':31,   'L_sup': 190},   # TAL -> HEL, Megastar
                                          1: {'N_trip': 1, 'N_ship': 2, 'L': 212, 'B':29,   'L_sup': 190},   # TAL -> STO, Baltic Queen
                                          2: {'N_trip': 1, 'N_ship': 2, 'L': 203, 'B':31.5, 'L_sup': 182},   # HEL -> STO, Serenade
                                          3: {'N_trip': 2, 'N_ship': 1, 'L': 212, 'B':29,   'L_sup': 190}},  # TUR -> STO, Baltic Princess
                          'uniform_fleet': False,
                          'present_fleet': True,
                          't_port_min': { 0: {0: 1, 1:1},
                                          1: {0: 7, 3:0.1, 4:7},
                                          2: {1: 7, 3:0.1, 4:7},
                                          3: {2: 1, 3:0.1, 4:1}}
                         })

  build_and_solve(ship_params, network_params)

# run_B4()

In [6]:
####################################
# Case U4: four routes, uniform fleet
####################################

def run_U4():
  ship_params = generate_ship_data()
  network_params = generate_network_data()

  network_params.update({ 'routes':{  0: [0, 1],        # TAL -> HEL
                                      1: [0, 3, 4],     # TAL -> MAR -> STO
                                      2: [1, 3, 4],     # HEL -> MAR -> STO
                                      3: [2, 3, 4]},    # TUR -> MAR -> STO
                          't_route': [36, 48, 48, 48],
                          'fixed_fleet': {},
                          'uniform_fleet': True,
                          'present_fleet': False,
                          't_port_min': {}
                         })

  build_and_solve(ship_params, network_params)

# run_U4()

In [7]:
####################################
# Case M4: four routes
####################################

def run_M4():
  ship_params = generate_ship_data()
  network_params = generate_network_data()

  network_params.update({ 'routes':{  0: [0, 1],        # TAL -> HEL
                                      1: [0, 3, 4],     # TAL -> MAR -> STO
                                      2: [1, 3, 4],     # HEL -> MAR -> STO
                                      3: [2, 3, 4]},    # TUR -> MAR -> STO
                          't_route': [36, 48, 48, 48],
                          'fixed_fleet': {},
                          'uniform_fleet': False,
                          'present_fleet': False,
                          't_port_min': {}
                         })

  build_and_solve(ship_params, network_params)

# run_M4()

In [8]:
####################################
# Case M3: three routes
####################################

def run_M3():
  ship_params = generate_ship_data()
  network_params = generate_network_data()

  network_params.update({ 'routes':{0: [0, 1],
                                    1: [0, 1, 3, 4],
                                    2: [2, 3, 4]},
                          't_route': [36, 48, 48],
                          'fixed_fleet': {},
                          'uniform_fleet': False,
                          'present_fleet': False,
                          't_port_min': {}
                         })

  build_and_solve(ship_params, network_params)

# run_M3()

In [9]:
####################################
# Case M2: two routes
####################################

def run_M2():
  ship_params = generate_ship_data()
  network_params = generate_network_data()

  network_params.update({ 'routes':{0: [0, 1, 3, 4],
                                    1: [2, 3, 4]},
                          't_route': [48, 48],
                          'fixed_fleet': {},
                          'uniform_fleet': False,
                          'present_fleet': False,
                          't_port_min': {}
                         })

  build_and_solve(ship_params, network_params)

# run_M2()

In [10]:
####################################
# Case M1: one route
####################################

def run_M1():
  ship_params = generate_ship_data()
  network_params = generate_network_data()

  network_params.update({ 'routes':{0: [0, 1, 2, 3, 4]},
                          't_route': [48],
                          'fixed_fleet': {},
                          'uniform_fleet': False,
                          'present_fleet': False,
                          't_port_min': {}
                         })

  build_and_solve(ship_params, network_params)

run_M1()

/usr/local/lib/python3.12/dist-packages/cvxpy/expressions/expression.py:498: FutureWarning: 
    You didn't specify the order of the flatten expression. The default order
    used in CVXPY is Fortran ('F') order. This default will change to match NumPy's
    default order ('C') in a future version of CVXPY.
    To suppress this warning, please specify the order explicitly.
    
  warnings.warn(flatten_order_warning, FutureWarning)


Status: optimal
Objective [kEUR]: 193769.0 
Charger powers [MW]:  ['Tal: 2.94', 'Hel: 19.05', 'Tur: 29.76', 'Mar: 26.65', 'Sto: 5.11']
Service 0: ['Tal->', 'Hel->', 'Tur->', 'Mar->', 'Sto->']
  N_trip: 1.00, N_ship: 14.00, E_batt [MWh]: 60.29, t_cell: 6.76, GT: 20487.5. L: 165.77, L_sup: 107.01, B: 23.51, T: 3.85, D: 10.44, p_td: 0.0181
  Speeds:  ['Tal->Hel: 17.54', 'Hel->Tur: 16.07', 'Tur->Hel: 16.07', 'Mar->Sto: 17.51', 'Sto->Mar: 17.46', 'Tur->Mar: 17.46', 'Hel->Tal: 17.50', 'Mar->Tur: 17.50']
